# NOTEBOOK 02
## PENGUMPULAN DATA DAN PENGOLAHAN AWAL KUALITAS UDARA

**Judul Penelitian:**
Pengelompokan Pola Konsentrasi Polutan Udara pada 38 Ibu Kota Provinsi di Indonesia Menggunakan Algoritma K-Means.

**Kelompok:** 5

Notebook ini berisi proses pengumpulan data kualitas udara dari 38 ibu kota provinsi di Indonesia menggunakan Open-Meteo Air Quality API dengan model CAMS Global.

Tahapan penelitian meliputi persiapan lokasi, pengumpulan data, audit, validasi, EDA, investigasi anomali, cleaning, agregasi harian, standardisasi, dan penyimpanan output.

## FASE 3 — PERSIAPAN DAN PENGUMPULAN DATA

In [ ]:

# ============================================
# FASE 3 - PERSIAPAN DAFTAR 38 LOKASI
# ============================================

import pandas as pd
from google.colab import files

# Upload file Excel koordinat
uploaded = files.upload()

nama_excel = next(
    name for name in uploaded.keys()
    if name.lower().endswith(".xlsx")
)

# Membaca sheet lokasi
df_lokasi = pd.read_excel(
    nama_excel,
    sheet_name="Lokasi_38",
    header=3
)

# Ambil baris yang memiliki data lokasi
kolom_wajib = [
    "No", "Provinsi", "Titik ibu kota",
    "Latitude", "Longitude", "Zona waktu"
]

df_lokasi = df_lokasi.dropna(
    subset=kolom_wajib
).copy()

# Konversi koordinat menjadi angka
df_lokasi["Latitude"] = pd.to_numeric(
    df_lokasi["Latitude"]
)

df_lokasi["Longitude"] = pd.to_numeric(
    df_lokasi["Longitude"]
)

# Konversi zona waktu Indonesia ke format API
peta_timezone = {
    "WIB": "Asia/Jakarta",
    "WITA": "Asia/Makassar",
    "WIT": "Asia/Jayapura"
}

df_lokasi["timezone_iana"] = (
    df_lokasi["Zona waktu"].map(peta_timezone)
)

# Pemeriksaan awal
if len(df_lokasi) != 38:
    raise ValueError(
        f"Seharusnya 38 lokasi, ditemukan {len(df_lokasi)}"
    )

if df_lokasi["timezone_iana"].isna().any():
    raise ValueError("Ada zona waktu yang belum dikenali.")

if df_lokasi[["Latitude", "Longitude"]].isna().any().any():
    raise ValueError("Ada koordinat yang kosong.")

df_lokasi = df_lokasi.sort_values("No").reset_index(drop=True)

print("DAFTAR LOKASI BERHASIL DIMUAT")
print("Jumlah lokasi:", len(df_lokasi))

display(
    df_lokasi[
        [
            "No", "Provinsi", "Titik ibu kota",
            "Latitude", "Longitude",
            "Zona waktu", "timezone_iana"
        ]
    ]
)


In [ ]:

# ============================================
# PENGAMBILAN DATASET 38 IBU KOTA PROVINSI
# OPEN-METEO AIR QUALITY API - CAMS GLOBAL
# ============================================

import requests
import pandas as pd
import numpy as np
import time

# 1. Konfigurasi penelitian
URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

TANGGAL_AWAL = "2026-08-20"
TANGGAL_AKHIR = "2026-11-17"

BATCH_SIZE = 10

FITUR = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

semua_data = []
metadata_lokasi = []

session = requests.Session()

# 2. Bagi 38 lokasi menjadi beberapa batch
for awal in range(0, len(df_lokasi), BATCH_SIZE):

    batch = df_lokasi.iloc[
        awal:awal + BATCH_SIZE
    ].copy()

    print(
        f"\nMengambil batch "
        f"{awal + 1} sampai {awal + len(batch)} "
        f"dari 38 lokasi..."
    )

    # 3. Siapkan parameter API
    params = {
        "latitude": ",".join(
            batch["Latitude"].astype(str)
        ),

        "longitude": ",".join(
            batch["Longitude"].astype(str)
        ),

        "hourly": ",".join(FITUR),

        "domains": "cams_global",

        "start_date": TANGGAL_AWAL,
        "end_date": TANGGAL_AKHIR,

        # Zona waktu tiap lokasi sesuai urutan koordinat
        "timezone": ",".join(
            batch["timezone_iana"].tolist()
        ),

        # Timestamp dikembalikan dalam Unix time / UTC
        "timeformat": "unixtime",

        "cell_selection": "land"
    }

    # 4. Hubungkan ke API
    response = session.get(
        URL,
        params=params,
        timeout=180
    )

    response.raise_for_status()

    hasil_api = response.json()

    if isinstance(hasil_api, dict):
        if hasil_api.get("error"):
            raise ValueError(
                hasil_api.get("reason")
            )

        hasil_api = [hasil_api]

    if len(hasil_api) != len(batch):
        raise ValueError(
            "Jumlah respons API tidak sesuai "
            "dengan jumlah lokasi pada batch."
        )

    # 5. Proses hasil setiap lokasi
    for (_, lokasi), hasil in zip(
        batch.iterrows(), hasil_api
    ):

        if hasil.get("error"):
            raise ValueError(
                f"Gagal untuk {lokasi['Titik ibu kota']}: "
                f"{hasil.get('reason')}"
            )

        hourly = hasil.get("hourly", {})

        if not hourly:
            raise ValueError(
                f"Data kosong: {lokasi['Titik ibu kota']}"
            )

        # Pastikan keenam fitur tersedia
        for fitur in FITUR:
            if fitur not in hourly:
                raise ValueError(
                    f"{fitur} tidak tersedia di "
                    f"{lokasi['Titik ibu kota']}"
                )

        # Ubah respons menjadi DataFrame
        df_kota = pd.DataFrame(hourly)

        # Unix timestamp -> datetime UTC
        waktu_unix = pd.to_numeric(
            df_kota.pop("time"),
            errors="raise"
        )

        df_kota["time_utc"] = pd.to_datetime(
            waktu_unix,
            unit="s",
            utc=True
        )

        # Offset lokal dari respons API
        offset = int(
            hasil.get("utc_offset_seconds", 0)
        )

        # Buat waktu lokal untuk analisis per kota
        df_kota["time_local"] = (
            df_kota["time_utc"]
            + pd.to_timedelta(offset, unit="s")
        ).dt.tz_localize(None)

        # Tambahkan identitas lokasi
        df_kota.insert(
            0, "no_provinsi", int(lokasi["No"])
        )

        df_kota.insert(
            1, "provinsi", lokasi["Provinsi"]
        )

        df_kota.insert(
            2, "kota", lokasi["Titik ibu kota"]
        )

        # Metadata koordinat dan sumber
        df_kota["zona_waktu"] = lokasi["Zona waktu"]
        df_kota["timezone_iana"] = lokasi["timezone_iana"]
        df_kota["utc_offset_seconds"] = offset

        df_kota["requested_latitude"] = lokasi["Latitude"]
        df_kota["requested_longitude"] = lokasi["Longitude"]

        # Koordinat grid model yang benar-benar dipakai
        df_kota["model_grid_latitude"] = hasil.get("latitude")
        df_kota["model_grid_longitude"] = hasil.get("longitude")

        df_kota["elevation_model"] = hasil.get("elevation")
        df_kota["model_source"] = "CAMS Global"

        semua_data.append(df_kota)

        # Simpan ringkasan per lokasi
        metadata_lokasi.append({
            "no": int(lokasi["No"]),
            "provinsi": lokasi["Provinsi"],
            "kota": lokasi["Titik ibu kota"],
            "requested_latitude": lokasi["Latitude"],
            "requested_longitude": lokasi["Longitude"],
            "timezone": lokasi["timezone_iana"],
            "utc_offset_seconds": offset,
            "model_grid_latitude": hasil.get("latitude"),
            "model_grid_longitude": hasil.get("longitude"),
            "jumlah_data": len(df_kota),
            "tanggal_awal_lokal": str(df_kota["time_local"].min()),
            "tanggal_akhir_lokal": str(df_kota["time_local"].max())
        })

    print("Batch berhasil:", len(batch), "lokasi")

    # Jeda kecil antarbatch
    time.sleep(2)

# 6. Gabungkan seluruh lokasi
df_raw = pd.concat(
    semua_data,
    ignore_index=True
)

df_raw = df_raw.sort_values(
    ["no_provinsi", "time_utc"]
).reset_index(drop=True)

df_metadata = pd.DataFrame(metadata_lokasi)

# 7. Simpan dataset mentah per jam
df_raw.to_csv(
    "dataset_kualitas_udara_raw.csv",
    index=False
)

print("\nPENGAMBILAN DATA SELESAI")
print("Total baris:", len(df_raw))
print("Total kota:", df_raw["kota"].nunique())

# 8. Ringkasan data per kota
ringkasan = df_raw.groupby("kota").agg(
    jumlah_pengamatan=("time_utc", "count"),
    tanggal_awal_utc=("time_utc", "min"),
    tanggal_akhir_utc=("time_utc", "max")
).reset_index()

display(ringkasan)

# 9. Periksa data kosong
print("\nDATA KOSONG:")
print(df_raw[FITUR].isna().sum())

# 10. Periksa duplikasi waktu
print("\nDUPLIKASI WAKTU PER KOTA:")
print(
    df_raw.duplicated(
        subset=["kota", "time_utc"]
    ).sum()
)

# 11. Simpan metadata lokasi
df_metadata.to_csv(
    "metadata_pengambilan_38_lokasi.csv",
    index=False
)


In [ ]:

# ============================================
# FILTER TIMESTAMP CAMS GLOBAL DAN AUDIT GRID
# ============================================

# Ambil timestamp setiap 3 jam berdasarkan UTC.
# Jangan menggunakan jam lokal untuk filter ini.

mask_3jam = (
    (df_raw["time_utc"].dt.minute == 0) &
    (df_raw["time_utc"].dt.hour % 3 == 0)
)

df_3jam = df_raw.loc[mask_3jam].copy()

# 1. Simpan data interval 3 jam
df_3jam.to_csv(
    "dataset_kualitas_udara_3jam.csv",
    index=False
)

# 2. Ringkasan jumlah data tiap kota
jumlah_per_kota = df_3jam.groupby("kota").size()

print("JUMLAH PENGAMATAN SETIAP 3 JAM:")
display(jumlah_per_kota)

print("\nTOTAL DATA 3 JAM:", len(df_3jam))

# 3. Audit grid model
grid = df_metadata.copy()

grid["grid_lat_round"] = (
    grid["model_grid_latitude"].round(4)
)

grid["grid_lon_round"] = (
    grid["model_grid_longitude"].round(4)
)

grid_audit = grid.groupby(
    ["grid_lat_round", "grid_lon_round"]
).agg(
    jumlah_lokasi=("kota", "nunique"),
    daftar_kota=("kota", lambda x: ", ".join(x))
).reset_index()

print("\nRINGKASAN GRID MODEL:")
display(grid_audit)

print("\nGRID YANG DIPAKAI LEBIH DARI SATU KOTA:")
display(
    grid_audit[
        grid_audit["jumlah_lokasi"] > 1
    ]
)

# Simpan hasil audit
grid_audit.to_csv(
    "audit_grid_model_38_lokasi.csv",
    index=False
)

# 4. Ringkasan kelengkapan enam polutan
print("\nDATA KOSONG PADA DATA 3 JAM:")
display(df_3jam[FITUR].isna().sum())

print("\nJUMLAH KOTA:", df_3jam["kota"].nunique())
print("JUMLAH BARIS:", len(df_3jam))


In [ ]:
from google.colab import files

files.download("dataset_kualitas_udara_raw.csv")
files.download("dataset_kualitas_udara_3jam.csv")
files.download("metadata_pengambilan_38_lokasi.csv")
files.download("audit_grid_model_38_lokasi.csv")

## FASE 4 — AUDIT DAN VALIDASI DATASET

Tahap ini dilakukan untuk memeriksa kelengkapan, konsistensi, duplikasi, missing value, nilai negatif, interval waktu, dan kesesuaian dataset interval tiga jam dengan data hasil pengambilan API.

In [ ]:

# ============================================
# FASE 4 - AUDIT KUALITAS DATASET
# ============================================

import pandas as pd
import numpy as np

FITUR = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

# 1. Buat salinan agar data mentah tidak berubah
df_audit = df_raw.copy()
df_audit_3jam = df_3jam.copy()

# 2. Pastikan format tanggal benar
df_audit["time_utc"] = pd.to_datetime(
    df_audit["time_utc"], utc=True
)

df_audit["time_local"] = pd.to_datetime(
    df_audit["time_local"]
)

df_audit_3jam["time_utc"] = pd.to_datetime(
    df_audit_3jam["time_utc"], utc=True
)

# 3. Pastikan fitur polutan berupa angka
for kolom in FITUR:
    df_audit[kolom] = pd.to_numeric(
        df_audit[kolom], errors="coerce"
    )

    df_audit_3jam[kolom] = pd.to_numeric(
        df_audit_3jam[kolom], errors="coerce"
    )

# ============================================
# A. UKURAN DATASET
# ============================================

print("A. UKURAN DATASET")

print("Data API per jam:", df_audit.shape)
print("Data interval 3 jam:", df_audit_3jam.shape)

print("Jumlah kota:", df_audit["kota"].nunique())
print("Jumlah provinsi:", df_audit["provinsi"].nunique())

# ============================================
# B. KELENGKAPAN DATA PER KOTA
# ============================================

print("\nB. KELENGKAPAN DATA PER KOTA")

ringkasan_kota = df_audit.groupby("kota").agg(
    jumlah_baris=("time_utc", "size"),
    jumlah_waktu=("time_utc", "nunique"),
    tanggal_awal=("time_local", "min"),
    tanggal_akhir=("time_local", "max"),
    jumlah_hari=(
        "time_local",
        lambda x: x.dt.date.nunique()
    )
).reset_index()

ringkasan_kota["target_90_hari"] = (
    ringkasan_kota["jumlah_hari"] == 90
)

display(ringkasan_kota)

# ============================================
# C. MISSING VALUE
# ============================================

print("\nC. MISSING VALUE")

missing_raw = df_audit[FITUR].isna().sum()
missing_3jam = df_audit_3jam[FITUR].isna().sum()

print("Data per jam:")
display(missing_raw.to_frame("jumlah_kosong"))

print("Data interval 3 jam:")
display(missing_3jam.to_frame("jumlah_kosong"))

# ============================================
# D. DUPLIKASI
# ============================================

print("\nD. DUPLIKASI")

duplikat_raw = df_audit.duplicated(
    subset=["kota", "time_utc"]
).sum()

duplikat_3jam = df_audit_3jam.duplicated(
    subset=["kota", "time_utc"]
).sum()

print("Duplikasi data per jam:", duplikat_raw)
print("Duplikasi data 3 jam:", duplikat_3jam)

# ============================================
# E. NILAI NEGATIF
# ============================================

print("\nE. PEMERIKSAAN NILAI NEGATIF")

negatif = (
    df_audit[FITUR] < 0
).sum()

display(negatif.to_frame("jumlah_nilai_negatif"))

# ============================================
# F. INTERVAL WAKTU
# ============================================

print("\nF. PEMERIKSAAN INTERVAL WAKTU")

for kota, grup in df_audit.groupby("kota"):

    interval = (
        grup.sort_values("time_utc")["time_utc"]
        .diff()
        .value_counts()
    )

    print(f"\n{kota}")
    print(interval.head())

print("\nInterval data 3 jam, contoh Medan:")

medan = df_audit_3jam[
    df_audit_3jam["kota"] == "Medan"
].sort_values("time_utc")

print(medan["time_utc"].diff().value_counts().head())

# ============================================
# G. GRID MODEL
# ============================================

print("\nG. PEMERIKSAAN GRID MODEL")

grid = df_metadata[
    [
        "kota",
        "model_grid_latitude",
        "model_grid_longitude"
    ]
].drop_duplicates()

grid["grid_lat"] = grid[
    "model_grid_latitude"
].round(4)

grid["grid_lon"] = grid[
    "model_grid_longitude"
].round(4)

audit_grid = grid.groupby(
    ["grid_lat", "grid_lon"]
).agg(
    jumlah_kota=("kota", "nunique"),
    daftar_kota=("kota", lambda x: ", ".join(x))
).reset_index()

display(audit_grid)

grid_sama = audit_grid[
    audit_grid["jumlah_kota"] > 1
]

print("\nGRID YANG DIGUNAKAN BEBERAPA KOTA:")
display(grid_sama)

# ============================================
# H. PEMERIKSAAN DATA 3 JAM
# ============================================

print("\nH. KESESUAIAN DATA 3 JAM DENGAN DATA API")

kunci_raw = df_audit[
    ["kota", "time_utc"]
].drop_duplicates()

cek_subset = df_audit_3jam[
    ["kota", "time_utc"]
].merge(
    kunci_raw,
    on=["kota", "time_utc"],
    how="left",
    indicator=True
)

print(
    "Data 3 jam yang ditemukan pada data per jam:",
    (cek_subset["_merge"] == "both").sum()
)

print(
    "Data 3 jam yang tidak ditemukan:",
    (cek_subset["_merge"] == "left_only").sum()
)

# ============================================
# I. RINGKASAN AKHIR
# ============================================

print("\n========== RINGKASAN AUDIT ==========")

print("Jumlah lokasi:", df_audit["kota"].nunique())
print("Jumlah baris API:", len(df_audit))
print("Jumlah baris 3 jam:", len(df_audit_3jam))
print("Jumlah missing value:", df_audit[FITUR].isna().sum().sum())
print("Jumlah duplikasi:", duplikat_raw)
print("Jumlah grid unik:", len(audit_grid))
print("Jumlah kelompok grid berulang:", len(grid_sama))

print("\nAUDIT AWAL SELESAI")


In [ ]:
# VALIDASI AKHIR DATASET 3 JAM

import numpy as np
import pandas as pd

FITUR = [
    "pm2_5", "pm10", "carbon_monoxide",
    "nitrogen_dioxide", "sulphur_dioxide", "ozone"
]

df_valid = df_3jam.copy()

# Pastikan tipe data polutan numerik
for kolom in FITUR:
    df_valid[kolom] = pd.to_numeric(
        df_valid[kolom], errors="coerce"
    )

# Periksa nilai kosong dan nilai tidak hingga
print("Tipe data:")
print(df_valid[FITUR].dtypes)

print("\nNilai kosong:")
print(df_valid[FITUR].isna().sum())

print("\nNilai tidak hingga:")
print(
    (~np.isfinite(df_valid[FITUR])).sum()
)

# Periksa kelengkapan interval setiap lokasi
hasil_validasi = []

for kota, grup in df_valid.groupby("kota"):

    grup = grup.sort_values("time_utc")

    waktu = pd.to_datetime(
        grup["time_utc"], utc=True
    )

    selisih = waktu.diff().dropna()

    hasil_validasi.append({
        "kota": kota,
        "jumlah_data": len(grup),
        "interval_3_jam": (
            selisih.eq(pd.Timedelta(hours=3)).all()
        ),
        "tanggal_awal": grup["time_local"].min(),
        "tanggal_akhir": grup["time_local"].max()
    })

hasil_validasi = pd.DataFrame(hasil_validasi)

display(hasil_validasi)

print("\nJumlah lokasi:", len(hasil_validasi))
print(
    "Lokasi dengan 720 data:",
    (hasil_validasi["jumlah_data"] == 720).sum()
)
print(
    "Lokasi dengan interval konsisten:",
    hasil_validasi["interval_3_jam"].sum()
)

# Simpan dataset tervalidasi
df_valid.to_csv(
    "dataset_kualitas_udara_clean_3jam.csv",
    index=False
)

hasil_validasi.to_csv(
    "laporan_validasi_38_lokasi.csv",
    index=False
)

print("\nDataset tervalidasi berhasil disimpan.")

## FASE 5 — EXPLORATORY DATA ANALYSIS (EDA)

Tahap ini bertujuan memahami karakteristik data, distribusi konsentrasi polutan, hubungan antarfitur, dan pola nilai ekstrem pada 38 lokasi penelitian.

In [ ]:
# FASE 5 - EDA AWAL
# Statistik deskriptif enam polutan

FITUR = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

# Statistik seluruh pengamatan
statistik = df_valid[FITUR].describe().T

statistik = statistik.rename(columns={
    "count": "jumlah_data",
    "mean": "rata_rata",
    "std": "standar_deviasi",
    "min": "minimum",
    "25%": "kuartil_1",
    "50%": "median",
    "75%": "kuartil_3",
    "max": "maksimum"
})

display(statistik.round(3))

# Statistik berdasarkan lokasi
statistik_kota = df_valid.groupby("kota")[FITUR].agg(
    ["mean", "median", "std", "min", "max"]
)

display(statistik_kota.round(3))

# Simpan hasil statistik
statistik.to_csv("statistik_deskriptif_global.csv")
statistik_kota.to_csv("statistik_deskriptif_per_lokasi.csv")

print("Statistik deskriptif berhasil dibuat.")

In [ ]:

# ============================================
# INVESTIGASI NILAI EKSTREM ENAM POLUTAN
# ============================================

FITUR = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

# 1. Sepuluh nilai tertinggi masing-masing polutan
for fitur in FITUR:

    print("\n================================")
    print("POLUTAN:", fitur)
    print("================================")

    kolom = [
        "kota",
        "provinsi",
        "time_utc",
        "time_local",
        fitur,
        "model_grid_latitude",
        "model_grid_longitude"
    ]

    display(
        df_valid.nlargest(10, fitur)[kolom]
    )

# 2. Periksa data khusus Palangka Raya
print("\n========== PALANGKA RAYA ==========")

palangka = df_valid[
    df_valid["kota"] == "Palangka Raya"
].copy()

display(
    palangka.sort_values(
        "pm2_5",
        ascending=False
    ).head(20)[
        ["time_utc", "time_local"] + FITUR
    ]
)

# 3. Periksa skewness setiap polutan
print("\n========== SKEWNESS ==========")

print(
    df_valid[FITUR].skew().round(3)
)

# 4. Hitung jumlah nilai di luar batas IQR
print("\n========== OUTLIER IQR ==========")

hasil_iqr = []

for fitur in FITUR:

    q1 = df_valid[fitur].quantile(0.25)
    q3 = df_valid[fitur].quantile(0.75)

    iqr = q3 - q1

    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr

    jumlah = (
        (df_valid[fitur] < batas_bawah) |
        (df_valid[fitur] > batas_atas)
    ).sum()

    hasil_iqr.append({
        "polutan": fitur,
        "batas_bawah": batas_bawah,
        "batas_atas": batas_atas,
        "jumlah_outlier": jumlah
    })

display(pd.DataFrame(hasil_iqr).round(3))


In [ ]:

# 1. PERIKSA PM2.5 > PM10

pm_tidak_wajar = df_valid[
    df_valid["pm2_5"] > df_valid["pm10"]
].copy()

print("JUMLAH PM2.5 > PM10:")
print(len(pm_tidak_wajar))

print("\nPERSENTASE:")
print(
    round(len(pm_tidak_wajar) / len(df_valid) * 100, 3),
    "%"
)

print("\nJUMLAH PER KOTA:")
display(
    pm_tidak_wajar.groupby("kota").size()
    .sort_values(ascending=False)
)

print("\n20 PENGAMATAN TERATAS:")
display(
    pm_tidak_wajar.sort_values(
        "pm2_5", ascending=False
    )[
        [
            "kota", "time_utc", "time_local",
            "pm2_5", "pm10"
        ]
    ].head(20)
)


In [ ]:

# 2. ANALISIS TREN PALANGKA RAYA

import matplotlib.pyplot as plt

palangka = df_valid[
    df_valid["kota"] == "Palangka Raya"
].copy()

palangka["tanggal"] = (
    palangka["time_local"].dt.date
)

# Rata-rata harian
harian = palangka.groupby("tanggal")[
    ["pm2_5", "pm10"]
].mean()

plt.figure(figsize=(14, 5))

plt.plot(
    harian.index,
    harian["pm2_5"],
    label="PM2.5"
)

plt.plot(
    harian.index,
    harian["pm10"],
    label="PM10"
)

plt.title(
    "Tren Rata-rata Harian PM2.5 dan PM10 - Palangka Raya"
)

plt.xlabel("Tanggal")
plt.ylabel("Konsentrasi (µg/m³)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Ringkasan harian tertinggi
display(
    harian.sort_values(
        "pm2_5",
        ascending=False
    ).head(10)
)


In [ ]:
# PERBANDINGAN RATA-RATA POLUTAN ANTARKOTA

perbandingan = df_valid.groupby("kota").agg(
    rata_pm25=("pm2_5", "mean"),
    median_pm25=("pm2_5", "median"),
    maksimum_pm25=("pm2_5", "max"),
    rata_pm10=("pm10", "mean"),
    maksimum_pm10=("pm10", "max")
).sort_values(
    "rata_pm25",
    ascending=False
)

display(perbandingan.round(2))

# Tampilkan 10 kota dengan rata-rata PM2.5 tertinggi
display(perbandingan.head(10).round(2))

In [ ]:

# ==========================================
# FASE 5 - EDA LANJUTAN
# KORELASI DAN DISTRIBUSI POLUTAN
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

FITUR = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

# 1. HEATMAP KORELASI
korelasi = df_valid[FITUR].corr(method="pearson")

plt.figure(figsize=(10, 7))

sns.heatmap(
    korelasi,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    center=0
)

plt.title("Korelasi Antarparameter Polutan")
plt.tight_layout()
plt.show()

# 2. DISTRIBUSI ENAM POLUTAN
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, fitur in zip(axes.flatten(), FITUR):

    # Transformasi hanya untuk tampilan grafik
    data_log = np.log1p(df_valid[fitur])

    sns.histplot(
        data_log,
        bins=40,
        kde=True,
        ax=ax
    )

    ax.set_title(f"Distribusi log1p: {fitur}")
    ax.set_xlabel("log(1 + konsentrasi)")

plt.tight_layout()
plt.show()

# 3. SKEWNESS SETELAH LOG1P
skew_awal = df_valid[FITUR].skew()

skew_log = np.log1p(
    df_valid[FITUR]
).skew()

hasil_skew = pd.DataFrame({
    "skewness_awal": skew_awal,
    "skewness_setelah_log1p": skew_log
})

print("PERBANDINGAN SKEWNESS")
display(hasil_skew.round(3))


In [ ]:

# KORELASI LEBIH DETAIL

korelasi = df_valid[FITUR].corr()

print("KORELASI PM2.5 DAN PM10:")
print(
    korelasi.loc["pm2_5", "pm10"]
)

print("\nSELURUH KORELASI:")
display(korelasi.style.format("{:.6f}"))

# Periksa selisih PM10 dan PM2.5
selisih = df_valid["pm10"] - df_valid["pm2_5"]

print("\nSTATISTIK SELISIH PM10 - PM2.5:")
display(selisih.describe())

# Scatter plot
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=df_valid,
    x="pm2_5",
    y="pm10",
    hue="kota",
    legend=False,
    alpha=0.4
)

plt.title("Hubungan PM2.5 dan PM10")
plt.xlabel("PM2.5 (µg/m³)")
plt.ylabel("PM10 (µg/m³)")
plt.tight_layout()
plt.show()


In [ ]:

# ==========================================
# KORELASI PM2.5 DAN PM10
# ==========================================

# 1. Korelasi seluruh data
print("KORELASI SELURUH DATA:")
print(
    df_valid["pm2_5"].corr(df_valid["pm10"])
)

# 2. Korelasi tanpa satu pengamatan anomali
df_pair = df_valid[
    df_valid["pm2_5"] <= df_valid["pm10"]
].copy()

print("\nKORELASI TANPA PENGAMATAN ANOMALI:")
print(
    df_pair["pm2_5"].corr(df_pair["pm10"])
)

# 3. Korelasi Pearson dan Spearman
print("\nPEARSON:")
print(
    df_pair["pm2_5"].corr(
        df_pair["pm10"], method="pearson"
    )
)

print("\nSPEARMAN:")
print(
    df_pair["pm2_5"].corr(
        df_pair["pm10"], method="spearman"
    )
)

# 4. Korelasi masing-masing kota
hasil_korelasi_kota = []

for kota, grup in df_valid.groupby("kota"):

    hasil_korelasi_kota.append({
        "kota": kota,
        "korelasi_pearson": grup[
            "pm2_5"
        ].corr(grup["pm10"])
    })

korelasi_kota = pd.DataFrame(
    hasil_korelasi_kota
).sort_values("korelasi_pearson", ascending=False)

print("\nKORELASI PM2.5 DAN PM10 PER KOTA:")
display(korelasi_kota.round(4))

print("\nRINGKASAN KORELASI PER KOTA:")
display(
    korelasi_kota["korelasi_pearson"].describe().round(4)
)


In [ ]:

# PERBANDINGAN TRANSFORMASI O3

from sklearn.preprocessing import PowerTransformer
import numpy as np
import pandas as pd

ozone = df_valid["ozone"].astype(float)

# Transformasi Yeo-Johnson
yeo = PowerTransformer(
    method="yeo-johnson",
    standardize=False
)

ozone_yeo = yeo.fit_transform(
    ozone.to_numpy().reshape(-1, 1)
).ravel()

hasil_transformasi = pd.DataFrame({
    "metode": [
        "Tanpa transformasi",
        "Log1p",
        "Square root",
        "Yeo-Johnson"
    ],
    "skewness": [
        ozone.skew(),
        np.log1p(ozone).skew(),
        np.sqrt(ozone).skew(),
        pd.Series(ozone_yeo).skew()
    ]
})

display(hasil_transformasi.round(4))


In [ ]:

# ==========================================
# PERBANDINGAN TRANSFORMASI SEMUA POLUTAN
# ==========================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import PowerTransformer

FITUR = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

data = df_valid[FITUR].astype(float).copy()

# Pastikan seluruh nilai dapat ditransformasi
if data.isna().any().any():
    raise ValueError("Masih ada nilai kosong.")

if (data < 0).any().any():
    raise ValueError(
        "Periksa nilai negatif sebelum log1p atau sqrt."
    )

# A. Tanpa transformasi
hasil_raw = data.copy()

# B. Log1p
hasil_log = np.log1p(data)

# C. Square root
hasil_sqrt = np.sqrt(data)

# D. Yeo-Johnson
yeo = PowerTransformer(
    method="yeo-johnson",
    standardize=False
)

hasil_yeo = pd.DataFrame(
    yeo.fit_transform(data),
    columns=FITUR,
    index=data.index
)

# 1. Hitung skewness setiap skenario
skewness = pd.DataFrame({
    "tanpa_transformasi": hasil_raw.skew(),
    "log1p": hasil_log.skew(),
    "square_root": hasil_sqrt.skew(),
    "yeo_johnson": hasil_yeo.skew()
})

print("PERBANDINGAN SKEWNESS ENAM POLUTAN:")
display(skewness.round(4))

# 2. Simpan hasil perbandingan
skewness.to_csv(
    "perbandingan_transformasi_polutan.csv"
)


## EKSPERIMEN ALTERNATIF — TRANSFORMASI YEO-JOHNSON

Bagian ini merupakan eksperimen pembanding untuk melihat pengaruh transformasi Yeo-Johnson terhadap distribusi data.

Eksperimen menggunakan lima fitur polutan dan tidak menggantikan baseline utama penelitian yang menggunakan enam fitur dengan StandardScaler.

In [ ]:

# ==========================================
# FASE 5 - FINALISASI KANDIDAT PREPROCESSING
# ==========================================

import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import PowerTransformer

# 1. Tentukan lima fitur utama
fitur_utama = [
    "pm2_5",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

# 2. Salin dataset agar data asli tidak berubah
df_phase5 = df_valid.copy()

# 3. Pastikan fitur numerik dan valid
X = df_phase5[fitur_utama].astype(float).copy()

if X.isna().any().any():
    raise ValueError("Masih terdapat missing value.")

if not np.isfinite(X.to_numpy()).all():
    raise ValueError("Terdapat nilai tidak hingga.")

if (X < 0).any().any():
    raise ValueError("Terdapat nilai negatif.")

# 4. Tandai anomali PM2.5 > PM10
df_phase5["flag_pm25_gt_pm10"] = (
    df_phase5["pm2_5"] > df_phase5["pm10"]
)

print(
    "Jumlah pengamatan PM2.5 > PM10:",
    df_phase5["flag_pm25_gt_pm10"].sum()
)

# 5. Terapkan transformasi Yeo-Johnson
transformer = PowerTransformer(
    method="yeo-johnson",
    standardize=True
)

X_transformed = transformer.fit_transform(X)

# 6. Buat DataFrame hasil transformasi
nama_fitur_transformasi = [
    "yj_" + fitur for fitur in fitur_utama
]

df_transformed = pd.DataFrame(
    X_transformed,
    columns=nama_fitur_transformasi,
    index=df_phase5.index
)

# 7. Gabungkan hasil dengan data asli
df_phase5 = pd.concat(
    [df_phase5, df_transformed],
    axis=1
)

# 8. Periksa hasil transformasi
print("\nSKEWNESS SETELAH TRANSFORMASI:")
display(
    df_phase5[nama_fitur_transformasi]
    .skew()
    .to_frame("skewness")
    .round(4)
)

print("\nRATA-RATA DAN STANDAR DEVIASI:")
display(
    df_phase5[nama_fitur_transformasi]
    .agg(["mean", "std"])
    .round(4)
)

# 9. Informasi ukuran dataset
print("\nUKURAN DATASET:")
print(df_phase5.shape)

print("Jumlah lokasi:", df_phase5["kota"].nunique())
print("Jumlah pengamatan:", len(df_phase5))

# 10. Simpan dataset kandidat
df_phase5.to_csv(
    "dataset_fase5_kandidat_preprocessing.csv",
    index=False
)

# 11. Simpan objek transformer
joblib.dump(
    transformer,
    "transformer_yeojohnson_5fitur.pkl"
)

print("\nFASE 5 BERHASIL DISIAPKAN.")
print("Dataset kandidat berhasil disimpan.")
print("Transformer berhasil disimpan.")


In [ ]:
from google.colab import files

files.download(
    "dataset_fase5_kandidat_preprocessing.csv"
)

files.download(
    "transformer_yeojohnson_5fitur.pkl"
)

## PIPELINE UTAMA — DATA CLEANING DAN PENGOLAHAN DATA HARIAN

Pipeline utama penelitian menggunakan enam fitur polutan. Dataset yang telah divalidasi akan melalui proses cleaning, agregasi harian, EDA data harian, dan standardisasi menggunakan StandardScaler.

In [ ]:

# ==========================================
# FASE 4 - FINALISASI DATA CLEANING
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path

# Folder output
BASE = Path("/content/output_fase4_5")
DATA_DIR = BASE / "02_DATASET"
EDA_DIR = BASE / "04_HASIL_ANALISIS" / "EDA"

DATA_DIR.mkdir(parents=True, exist_ok=True)
EDA_DIR.mkdir(parents=True, exist_ok=True)

FITUR = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

# Salin dataset yang sudah melalui validasi akhir
df_clean = df_valid.copy()

# Periksa kolom wajib
kolom_wajib = [
    "kota", "provinsi", "time_utc",
    "time_local"
] + FITUR

kolom_hilang = [
    kolom for kolom in kolom_wajib
    if kolom not in df_clean.columns
]

if kolom_hilang:
    raise ValueError(
        f"Kolom tidak ditemukan: {kolom_hilang}"
    )

# Pastikan timestamp benar
df_clean["time_utc"] = pd.to_datetime(
    df_clean["time_utc"],
    utc=True,
    errors="coerce"
)

df_clean["time_local"] = pd.to_datetime(
    df_clean["time_local"],
    errors="coerce"
)

# Pastikan enam fitur numerik
for kolom in FITUR:
    df_clean[kolom] = pd.to_numeric(
        df_clean[kolom],
        errors="coerce"
    )

# Audit kualitas
missing = df_clean[kolom_wajib].isna().sum()

duplikasi = df_clean.duplicated(
    subset=["kota", "time_utc"]
).sum()

negatif = (df_clean[FITUR] < 0).sum()

tidak_finit = (
    ~np.isfinite(df_clean[FITUR].to_numpy())
).sum()

# Hentikan proses jika ada masalah mendasar
if missing.sum() > 0:
    raise ValueError(
        "Masih terdapat missing value. "
        "Periksa dahulu sebelum menyimpan dataset."
    )

if duplikasi > 0:
    raise ValueError(
        "Masih terdapat duplikasi waktu."
    )

if negatif.sum() > 0:
    raise ValueError(
        "Masih terdapat nilai konsentrasi negatif."
    )

if tidak_finit > 0:
    raise ValueError(
        "Terdapat nilai inf atau tidak hingga."
    )

# Tandai anomali PM2.5 > PM10
df_clean["flag_pm25_gt_pm10"] = (
    df_clean["pm2_5"] > df_clean["pm10"]
)

jumlah_anomali = int(
    df_clean["flag_pm25_gt_pm10"].sum()
)

# Simpan dataset bersih
path_clean = DATA_DIR / "dataset_kualitas_udara_clean.csv"

df_clean.to_csv(path_clean, index=False)

# Buat catatan cleaning
catatan = f"""
CATATAN DATA CLEANING
PROYEK KUALITAS UDARA - KELOMPOK 5

Sumber data:
Open-Meteo Air Quality API
Model: CAMS Global

Periode:
20 Agustus 2026 sampai 17 November 2026

Jumlah lokasi:
{df_clean['kota'].nunique()}

Jumlah pengamatan:
{len(df_clean)}

Interval:
3 jam berdasarkan timestamp UTC

Jumlah fitur polutan:
6

HASIL PEMERIKSAAN:
Missing value: {int(missing.sum())}
Duplikasi waktu per kota: {duplikasi}
Nilai negatif: {int(negatif.sum())}
Nilai tidak hingga: {tidak_finit}
Anomali PM2.5 > PM10: {jumlah_anomali}

KEPUTUSAN CLEANING:
1. Tipe timestamp dan fitur numerik diperiksa.
2. Tidak dilakukan imputasi karena tidak ditemukan missing value.
3. Tidak dilakukan penghapusan baris duplikat karena tidak ditemukan duplikasi.
4. Tidak ditemukan nilai konsentrasi negatif.
5. Anomali PM2.5 > PM10 diberi penanda dan dipertahankan.
6. Nilai ekstrem tidak dihapus secara otomatis.
7. Dataset mentah tetap dipertahankan sebagai sumber asli.

Dataset output:
dataset_kualitas_udara_clean.csv
"""

path_catatan = BASE / "catatan_cleaning.txt"
path_catatan.write_text(catatan, encoding="utf-8")

print("FASE 4 BERHASIL")
print("Jumlah baris:", len(df_clean))
print("Jumlah kota:", df_clean["kota"].nunique())
print("Anomali yang ditandai:", jumlah_anomali)
print("File:", path_clean)
print("Catatan:", path_catatan)


In [ ]:

# ==========================================
# FASE 5 - AGREGASI DATA HARIAN
# ==========================================

# Tambahkan tanggal lokal
df_work = df_clean.copy()

df_work["tanggal"] = (
    df_work["time_local"].dt.date
)

KEYS = ["provinsi", "kota", "tanggal"]

# Hitung jumlah pengamatan dan anomali per hari
audit_harian = df_work.groupby(KEYS).agg(
    jumlah_pengamatan=("time_utc", "count"),
    jumlah_anomali=("flag_pm25_gt_pm10", "sum")
).reset_index()

# Setiap hari harus memiliki 8 data interval 3 jam
hari_lengkap = audit_harian[
    audit_harian["jumlah_pengamatan"] == 8
].copy()

hari_tidak_lengkap = audit_harian[
    audit_harian["jumlah_pengamatan"] != 8
].copy()

print("AUDIT KELENGKAPAN DATA HARIAN")
print("Hari lengkap:", len(hari_lengkap))
print("Hari tidak lengkap:", len(hari_tidak_lengkap))

# Simpan laporan audit harian
audit_harian.to_csv(
    DATA_DIR / "audit_kelengkapan_harian.csv",
    index=False
)

# Ambil hanya hari yang lengkap
df_lengkap = df_work.merge(
    hari_lengkap[KEYS],
    on=KEYS,
    how="inner"
)

# Hitung rata-rata keenam polutan
aturan_agregasi = {
    fitur: "mean" for fitur in FITUR
}

aturan_agregasi.update({
    "time_utc": "count",
    "flag_pm25_gt_pm10": "sum"
})

df_harian = df_lengkap.groupby(
    KEYS
).agg(
    aturan_agregasi
).reset_index()

df_harian = df_harian.rename(
    columns={
        "time_utc": "jumlah_pengamatan",
        "flag_pm25_gt_pm10": "jumlah_anomali"
    }
)

# Urutkan agar metadata dan matriks fitur sejajar
df_harian = df_harian.sort_values(
    ["provinsi", "kota", "tanggal"]
).reset_index(drop=True)

# Berikan ID setiap baris
df_harian.insert(
    0,
    "id_pengamatan",
    range(1, len(df_harian) + 1)
)

# Simpan dataset harian lengkap
df_harian.to_csv(
    DATA_DIR / "dataset_harian.csv",
    index=False
)

# Simpan metadata secara terpisah
kolom_metadata = [
    "id_pengamatan",
    "provinsi",
    "kota",
    "tanggal",
    "jumlah_pengamatan",
    "jumlah_anomali"
]

df_metadata_harian = df_harian[
    kolom_metadata
].copy()

df_metadata_harian.to_csv(
    DATA_DIR / "metadata_harian.csv",
    index=False
)

print("\nDATASET HARIAN BERHASIL DIBUAT")
print("Jumlah baris:", len(df_harian))
print("Jumlah kota:", df_harian["kota"].nunique())

display(df_harian.head(10))


In [ ]:

# ==========================================
# FASE 5 - PENYIMPANAN GRAFIK EDA
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

# 1. Statistik deskriptif global
statistik_global = df_harian[FITUR].describe().T

statistik_global.to_csv(
    EDA_DIR / "statistik_deskriptif_global.csv"
)

# 2. Statistik deskriptif per kota
statistik_kota = df_harian.groupby("kota")[FITUR].agg(
    ["mean", "median", "std", "min", "max"]
)

statistik_kota.to_csv(
    EDA_DIR / "statistik_deskriptif_per_lokasi.csv"
)

# 3. Heatmap korelasi
korelasi_harian = df_harian[FITUR].corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    korelasi_harian,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Korelasi Enam Parameter Polutan - Data Harian")
plt.tight_layout()

plt.savefig(
    EDA_DIR / "heatmap_korelasi.png",
    dpi=300
)
plt.show()
plt.close()

# 4. Boxplot keenam polutan
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, fitur in zip(axes.flatten(), FITUR):
    sns.boxplot(
        y=df_harian[fitur],
        ax=ax
    )
    ax.set_title(fitur)
    ax.set_ylabel("Konsentrasi (µg/m³)")

fig.suptitle("Boxplot Enam Parameter Polutan")
plt.tight_layout()

plt.savefig(
    EDA_DIR / "boxplot_polutan.png",
    dpi=300
)
plt.show()
plt.close()

# 5. Distribusi keenam polutan
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, fitur in zip(axes.flatten(), FITUR):
    sns.histplot(
        df_harian[fitur],
        bins=40,
        kde=True,
        ax=ax
    )
    ax.set_title(f"Distribusi {fitur}")
    ax.set_xlabel("Konsentrasi (µg/m³)")

fig.suptitle("Distribusi Enam Parameter Polutan")
plt.tight_layout()

plt.savefig(
    EDA_DIR / "distribusi_polutan.png",
    dpi=300
)
plt.show()
plt.close()

# 6. Grafik tren rata-rata harian PM2.5
tren = df_harian.groupby("tanggal")[
    "pm2_5"
].mean()

plt.figure(figsize=(14, 5))

plt.plot(
    pd.to_datetime(tren.index),
    tren.values,
    linewidth=2
)

plt.title("Tren Rata-rata PM2.5 pada 38 Lokasi")
plt.xlabel("Tanggal")
plt.ylabel("PM2.5 (µg/m³)")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    EDA_DIR / "tren_pm25_lokasi.png",
    dpi=300
)
plt.show()
plt.close()

# 7. Perbandingan rata-rata PM2.5 setiap kota
perbandingan = df_harian.groupby("kota")[
    "pm2_5"
].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 9))

perbandingan.sort_values().plot(
    kind="barh"
)

plt.title("Perbandingan Rata-rata PM2.5 pada 38 Lokasi")
plt.xlabel("PM2.5 (µg/m³)")
plt.ylabel("Lokasi")
plt.tight_layout()

plt.savefig(
    EDA_DIR / "perbandingan_pm25_kota.png",
    dpi=300
)
plt.show()
plt.close()

# 8. Simpan korelasi
korelasi_harian.to_csv(
    EDA_DIR / "korelasi_polutan.csv"
)

print("SELURUH OUTPUT EDA BERHASIL DISIMPAN")


In [ ]:

# ==========================================
# FASE 5 - STANDARDISASI DATA
# ==========================================

from sklearn.preprocessing import StandardScaler
import joblib

# Ambil enam fitur dari dataset harian
X = df_harian[FITUR].copy()

# Pastikan tidak ada data kosong
if X.isna().any().any():
    raise ValueError("Masih ada nilai kosong.")

# Standardisasi
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Buat DataFrame
df_standard = pd.DataFrame(
    X_scaled,
    columns=FITUR
)

# ID disimpan sebagai penanda baris,
# bukan sebagai fitur untuk K-Means.
df_standard.insert(
    0,
    "id_pengamatan",
    df_harian["id_pengamatan"].values
)

# Simpan matriks terstandardisasi
df_standard.to_csv(
    DATA_DIR / "data_terstandardisasi.csv",
    index=False
)

# Simpan objek scaler untuk reproduksi
joblib.dump(
    scaler,
    DATA_DIR / "standard_scaler.pkl"
)

print("DATA STANDARDISASI BERHASIL")

print("\nRata-rata setiap fitur:")
print(df_standard[FITUR].mean().round(4))

print("\nStandar deviasi setiap fitur:")
print(df_standard[FITUR].std().round(4))

display(df_standard.head(10))


## PENYIMPANAN DAN PENGEMASAN OUTPUT FASE 4–5

Seluruh hasil pengolahan Fase 4 dan Fase 5 dikumpulkan menjadi satu arsip ZIP agar mudah disimpan dan digunakan pada tahapan pemodelan K-Means berikutnya.

In [ ]:

# ==========================================
# PAKET OUTPUT FASE 4 DAN FASE 5
# ==========================================

import shutil
from google.colab import files

# Buat arsip ZIP
zip_path = shutil.make_archive(
    "/content/OUTPUT_FASE_4_5",
    "zip",
    root_dir="/content",
    base_dir="output_fase4_5"
)

print("PAKET BERHASIL DIBUAT:")
print(zip_path)

# Download ke laptop
files.download(zip_path)
